# SpatialMesh — Day 8: GAT-GNN Model
**Goal:** Build the Graph Attention Network that maps the speaker graph → (az, el) per speaker

**Key decisions:**
- `GATv2Conv` with `edge_dim=7` — edge features participate in attention (plain GATConv ignores them)
- 2 layers, 4 heads, residual refinement of current position
- Output [4, 2] normalized [-1, 1], tanh-bounded

**Verifies:** forward pass → backward pass with `spatialmesh_loss` → loss decreases over steps

In [ ]:
# Cell 1 — Install PyG (no compiled extensions; small graphs)
!pip install -q torch_geometric
import torch_geometric
print(f'PyG version: {torch_geometric.__version__}')

In [ ]:
# Cell 2 — Imports + mount
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from google.colab import drive
drive.mount('/content/drive')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Locked constants
NODE_DIM, EDGE_DIM = 133, 7
HIDDEN, HEADS, OUT_DIM = 64, 4, 2
N_SPEAKERS = 4
print(f'Device: {DEVICE}')

In [ ]:
# Cell 3 — Loss function (paste spatialmesh_loss.py contents OR import)
# If you saved spatialmesh_loss.py to Drive src/, load it:
import sys
sys.path.append('/content/drive/MyDrive/SpatialMesh/src/')
from spatialmesh_loss import spatialmesh_loss, perceptual_separability_score
print('Loss function imported.')

In [ ]:
# Cell 4 — GAT-GNN Model Definition
class SpatialMeshGAT(nn.Module):
    """
    2-layer Graph Attention Network with edge features.
    GATv2Conv(133→64, 4 heads, concat) → GATv2Conv(256→32, 4 heads, avg)
    → Linear head → tanh → residual refinement of current position.
    """
    def __init__(self, node_dim=NODE_DIM, edge_dim=EDGE_DIM,
                 hidden=HIDDEN, heads=HEADS, out_dim=OUT_DIM,
                 refine=True, refine_gain=0.5):
        super().__init__()
        self.refine = refine
        self.refine_gain = refine_gain

        self.gat1 = GATv2Conv(node_dim, hidden, heads=heads,
                              concat=True, edge_dim=edge_dim,
                              dropout=0.1, add_self_loops=False)
        self.norm1 = nn.LayerNorm(hidden * heads)

        self.gat2 = GATv2Conv(hidden * heads, 32, heads=heads,
                              concat=False, edge_dim=edge_dim,
                              dropout=0.1, add_self_loops=False)
        self.norm2 = nn.LayerNorm(32)

        self.head = nn.Sequential(
            nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, out_dim)
        )

    def forward(self, x, edge_index, edge_attr):
        current_pos = x[:, 131:133]
        h = F.elu(self.norm1(self.gat1(x, edge_index, edge_attr)))
        h = F.elu(self.norm2(self.gat2(h, edge_index, edge_attr)))
        delta = torch.tanh(self.head(h))
        if self.refine:
            return torch.tanh(current_pos + self.refine_gain * delta)
        return delta


model = SpatialMeshGAT().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')
print(f'Est. size:  {n_params * 4 / 1024**2:.3f} MB')

In [ ]:
# Cell 5 — Load Day 7 example graph + forward pass
GRAPH_PATH = '/content/drive/MyDrive/SpatialMesh/graphs/example_graph.pt'
g = torch.load(GRAPH_PATH, map_location=DEVICE)

x          = g['x'].to(DEVICE)
edge_index = g['edge_index'].to(DEVICE)
edge_attr  = g['edge_attr'].to(DEVICE)

print(f'Loaded graph: x{tuple(x.shape)} '
      f'edge_index{tuple(edge_index.shape)} '
      f'edge_attr{tuple(edge_attr.shape)}')

model.eval()
with torch.no_grad():
    out = model(x, edge_index, edge_attr)
print(f'\nOutput shape: {tuple(out.shape)}')
print(f'Output range: [{out.min():.3f}, {out.max():.3f}]')
print(f'Predicted positions (az, el normalized):')
for i in range(N_SPEAKERS):
    az = out[i,0].item()*180; el = out[i,1].item()*90
    print(f'  Speaker {i}: az={az:+7.1f}°  el={el:+6.1f}°')

assert out.shape == (N_SPEAKERS, OUT_DIM)
assert out.min() >= -1.0 and out.max() <= 1.0
print('\n✅ Forward pass valid.')

In [ ]:
# Cell 6 — Backward pass with loss
model.train()
out = model(x, edge_index, edge_attr)

loss, comp = spatialmesh_loss(
    out,
    g['prev_positions'].to(DEVICE),
    g['embeddings'].to(DEVICE),
    g['activity_mask'].to(DEVICE),
    g['dominance'].to(DEVICE),
    g['overlap_duration'].to(DEVICE),
)
loss.backward()

print('Loss components:')
for k, v in comp.items():
    print(f'  {k:14s}: {v}')

has_grad = all(p.grad is not None for p in model.parameters() if p.requires_grad)
finite   = all(torch.isfinite(p.grad).all().item()
               for p in model.parameters() if p.grad is not None)
print(f'\nGradients flow: {has_grad} | all finite: {finite}')
assert has_grad and finite
print('✅ Backward pass valid.')

In [ ]:
# Cell 7 — Overfit sanity: loss must drop on a single graph
model_test = SpatialMeshGAT().to(DEVICE)
opt = torch.optim.Adam(model_test.parameters(), lr=1e-3)

print('Overfitting single graph (loss should fall steadily):')
for step in range(100):
    opt.zero_grad()
    pred = model_test(x, edge_index, edge_attr)
    l, c = spatialmesh_loss(
        pred, g['prev_positions'].to(DEVICE), g['embeddings'].to(DEVICE),
        g['activity_mask'].to(DEVICE), g['dominance'].to(DEVICE),
        g['overlap_duration'].to(DEVICE))
    l.backward()
    opt.step()
    if step % 20 == 0:
        sep = perceptual_separability_score(
            pred, g['embeddings'].to(DEVICE), g['activity_mask'].to(DEVICE),
            g['dominance'].to(DEVICE), g['overlap_duration'].to(DEVICE))
        print(f'  step {step:3d}: loss {l.item():.4f}  separability {sep:.1f}')

print('\n✅ If loss fell and separability rose, the model + loss are wired correctly.')
print('Day 8 architecture verified. Ready for Day 9 data generation.')

In [ ]:
# Cell 8 — Save model definition + untrained checkpoint
import json
SAVE_DIR = '/content/drive/MyDrive/SpatialMesh/models/'

torch.save(model.state_dict(), SAVE_DIR + 'gat_init.pt')

config = {
    'node_dim': NODE_DIM, 'edge_dim': EDGE_DIM,
    'hidden': HIDDEN, 'heads': HEADS, 'out_dim': OUT_DIM,
    'refine': True, 'refine_gain': 0.5,
    'n_params': n_params,
    'conv_type': 'GATv2Conv',
}
with open(SAVE_DIR + 'gat_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f'Saved gat_init.pt and gat_config.json to {SAVE_DIR}')
print('=== Day 8 Complete ===')